- Ce script charge d'abord un fichier CSV nommé Data_EHCVM_2021.csv, qui contient des données de ménages incluant les coordonnées GPS, et l'utilise pour chercher des correspondances avec des images satellites localisées dans un dossier spécifique. Les images satellites sont nommées avec des informations de latitude et de longitude, et le script analyse les noms de ces fichiers pour en extraire ces coordonnées. Ensuite, il compare ces valeurs extraites avec les colonnes gps__latitude et gps__longitude dans le fichier CSV pour identifier les ménages correspondant aux images disponibles.

- Pour chaque image ayant une correspondance, le script récupère les informations complètes du ménage depuis le fichier CSV et crée une entrée avec le nom de l'image en plus des autres données. Les colonnes sont ensuite réorganisées pour placer en première position le nom de l'image, suivi de colonnes clés (gps__latitude, gps__Longitude, grappe, region, hhweight, hhsize, pcexp) avant de compléter avec toutes les autres colonnes restantes.

- Enfin, le script sauvegarde le nouveau DataFrame dans un dossier spécifié. Si ce dossier n'existe pas, il est créé automatiquement. Le fichier résultant, nommé Data_EHCVM_2021_images.csv, contient donc une vue consolidée et organisée des informations de ménages associés à leurs images satellites.

In [1]:
import pandas as pd
import os
import glob

# Charger le fichier Data_EHCVM_2021.csv
data_path = r"D:\wealth_predict_2021\data\original_csv_file\Data_EHCVM_2021.csv"
data = pd.read_csv(data_path)

# Définir le chemin du dossier des images
images_folder = r"D:\wealth_predict_2021\data\downloaded\Image_satellite_EHCVM_2021_Zoom_18_Image_2024"

# Chercher les images avec plusieurs extensions possibles
image_extensions = ["*.jpeg", "*.png", "*.tif", "*.tiff"] 
images = []
for ext in image_extensions:
    images.extend(glob.glob(os.path.join(images_folder, ext)))

# Vérifier si des images ont été trouvées
if not images:
    print("Aucune image trouvée dans le dossier spécifié.")
else:
    # Créer une liste pour stocker les informations
    image_data = []

    # Extraire latitude et longitude des noms de fichier et associer avec les données de DataCIV3
    for image_path in images:
        # Extraire le nom de l'image
        image_name = os.path.basename(image_path)
        
        # Extraire latitude et longitude depuis le nom de l'image
        try:
            parts = image_name.split("_")
            latitude = float(parts[1])
            longitude = float(parts[3])
            
            # Filtrer Data_EHCVM_2021 pour correspondre les GPS
            matching_row = data[(data['gps__latitude'] == latitude) & (data['gps__longitude'] == longitude)]
            
            # Si une correspondance est trouvée, ajouter les informations au DataFrame
            if not matching_row.empty:
                # Ajouter le nom de l'image et les autres colonnes au DataFrame
                row_data = matching_row.iloc[0].to_dict()
                row_data["nom de l'image"] = image_name
                image_data.append(row_data)
                
        except (IndexError, ValueError):
            print(f"Erreur lors du traitement de l'image {image_name}. Nom de fichier non valide.")

    # Créer un DataFrame avec les informations des images
    image_df = pd.DataFrame(image_data)

    # Réorganiser les colonnes
    columns_order = ['nom de l\'image', 'gps__latitude', 'gps__longitude', 'grappe', 'region', 
                     'hhweight', 'hhsize', 'pcexp']
    remaining_columns = [col for col in image_df.columns if col not in columns_order]
    ordered_columns = columns_order + remaining_columns
    image_df = image_df[ordered_columns]

    # Définir le chemin du dossier de sauvegarde et créer le dossier s'il n'existe pas
    output_folder = r'D:\wealth_predict_2021\data\processed_csv'
    os.makedirs(output_folder, exist_ok=True)

    # Sauvegarder le DataFrame
    output_path = os.path.join(output_folder, "Data_EHCVM_2021_images.csv") 
    image_df.to_csv(output_path, index=False)

    print(f"DataFrame sauvegardé avec succès dans {output_path}")


DataFrame sauvegardé avec succès dans D:\wealth_predict_2021\data\processed_csv\Data_EHCVM_2021_images.csv


In [2]:
image_df

,nom de l'image,gps__latitude,gps__longitude,grappe,region,hhweight,hhsize,pcexp,country,year,...,hbranch,hsectins,hcsp,dali,dnal,dtot,zref,def_spa,def_temp,pcexp_binaire
0,lat_10.00204853_lon_-5.93779182_zoom_18.jpeg,10.002049,-5.937792,168,PORO,327.53464,5,481576.38,CIV,2021,...,Agriculture,Entreprise Privée,Travailleur pour compte propre,1.198395e+06,1.078191e+06,2276586.0,369516.44,0.945472,1.026043,0
1,lat_10.00730657_lon_-5.90204798_zoom_18.jpeg,10.007307,-5.902048,167,PORO,597.89716,1,1476875.20,CIV,2021,...,Trans./Comm.,Entreprise Privée,Travailleur pour compte propre,4.018170e+05,1.023466e+06,1425282.9,369516.44,0.965067,0.975638,0
2,lat_10.00738169_lon_-5.90098456_zoom_18.jpeg,10.007382,-5.900985,167,PORO,597.89716,5,449456.53,CIV,2021,...,Agriculture,Entreprise Privée,Travailleur pour compte propre,1.424693e+06,7.440838e+05,2168777.2,369516.44,0.965067,0.975638,0
3,lat_10.00749888_lon_-5.90327854_zoom_18.jpeg,10.007499,-5.903279,167,PORO,597.89716,4,330621.00,CIV,2021,...,NaN,NaN,NaN,8.259347e+05,4.503503e+05,1276285.0,369516.44,0.965067,0.975638,1
4,lat_10.00786655_lon_-5.90329469_zoom_18.jpeg,10.007867,-5.903295,167,PORO,713.73000,10,372874.88,CIV,2021,...,Elevage/syl./peche,Entreprise Privée,Patron/Employeur,2.388965e+06,1.209526e+06,3598490.5,369516.44,0.965067,0.975638,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12947,lat_9.99845391_lon_-5.57904719_zoom_18.jpeg,9.998454,-5.579047,1047,TCHOLOGO,327.80972,2,849016.40,CIV,2021,...,Agriculture,Entreprise Privée,Travailleur pour compte propre,9.347838e+05,6.706593e+05,1605443.1,369516.44,0.945472,0.971834,0
12948,lat_9.9984914_lon_-7.8351835_zoom_18.jpeg,9.998491,-7.835184,800,FOLON,117.29432,4,387448.16,CIV,2021,...,Commerce,Entreprise Privée,Travailleur pour compte propre,7.550804e+05,7.405725e+05,1495652.9,369516.44,0.965067,1.023907,0
12949,lat_9.99851123_lon_-5.57863867_zoom_18.jpeg,9.998511,-5.578639,1047,TCHOLOGO,203.61246,1,365787.10,CIV,2021,...,NaN,NaN,NaN,2.027978e+05,1.430438e+05,345841.6,369516.44,0.945472,0.971834,1
12950,lat_9.99976359_lon_-5.57897807_zoom_18.jpeg,9.999764,-5.578978,1047,TCHOLOGO,203.61246,7,223828.75,CIV,2021,...,Commerce,Entreprise Privée,"Manœuvre, aide ménagère",8.971284e+05,5.842390e+05,1481367.4,369516.44,0.945472,0.971834,1


In [3]:
pd.read_csv(r"D:\wealth_predict_2021\data\processed_csv\Data_EHCVM_2021_images.csv")

,nom de l'image,gps__latitude,gps__longitude,grappe,region,hhweight,hhsize,pcexp,country,year,...,hbranch,hsectins,hcsp,dali,dnal,dtot,zref,def_spa,def_temp,pcexp_binaire
0,lat_10.00204853_lon_-5.93779182_zoom_18.jpeg,10.002049,-5.937792,168,PORO,327.53464,5,481576.38,CIV,2021,...,Agriculture,Entreprise Privée,Travailleur pour compte propre,1.198395e+06,1.078191e+06,2276586.0,369516.44,0.945472,1.026043,0
1,lat_10.00730657_lon_-5.90204798_zoom_18.jpeg,10.007307,-5.902048,167,PORO,597.89716,1,1476875.20,CIV,2021,...,Trans./Comm.,Entreprise Privée,Travailleur pour compte propre,4.018170e+05,1.023466e+06,1425282.9,369516.44,0.965067,0.975638,0
2,lat_10.00738169_lon_-5.90098456_zoom_18.jpeg,10.007382,-5.900985,167,PORO,597.89716,5,449456.53,CIV,2021,...,Agriculture,Entreprise Privée,Travailleur pour compte propre,1.424693e+06,7.440838e+05,2168777.2,369516.44,0.965067,0.975638,0
3,lat_10.00749888_lon_-5.90327854_zoom_18.jpeg,10.007499,-5.903279,167,PORO,597.89716,4,330621.00,CIV,2021,...,NaN,NaN,NaN,8.259347e+05,4.503503e+05,1276285.0,369516.44,0.965067,0.975638,1
4,lat_10.00786655_lon_-5.90329469_zoom_18.jpeg,10.007867,-5.903295,167,PORO,713.73000,10,372874.88,CIV,2021,...,Elevage/syl./peche,Entreprise Privée,Patron/Employeur,2.388965e+06,1.209526e+06,3598490.5,369516.44,0.965067,0.975638,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12947,lat_9.99845391_lon_-5.57904719_zoom_18.jpeg,9.998454,-5.579047,1047,TCHOLOGO,327.80972,2,849016.40,CIV,2021,...,Agriculture,Entreprise Privée,Travailleur pour compte propre,9.347838e+05,6.706593e+05,1605443.1,369516.44,0.945472,0.971834,0
12948,lat_9.9984914_lon_-7.8351835_zoom_18.jpeg,9.998491,-7.835184,800,FOLON,117.29432,4,387448.16,CIV,2021,...,Commerce,Entreprise Privée,Travailleur pour compte propre,7.550804e+05,7.405725e+05,1495652.9,369516.44,0.965067,1.023907,0
12949,lat_9.99851123_lon_-5.57863867_zoom_18.jpeg,9.998511,-5.578639,1047,TCHOLOGO,203.61246,1,365787.10,CIV,2021,...,NaN,NaN,NaN,2.027978e+05,1.430438e+05,345841.6,369516.44,0.945472,0.971834,1
12950,lat_9.99976359_lon_-5.57897807_zoom_18.jpeg,9.999764,-5.578978,1047,TCHOLOGO,203.61246,7,223828.75,CIV,2021,...,Commerce,Entreprise Privée,"Manœuvre, aide ménagère",8.971284e+05,5.842390e+05,1481367.4,369516.44,0.945472,0.971834,1
